# Basics of Mobile Robotics - Project Report

## Filtering

To ensure robust position estimation of the Thymio robot in the environment, we implemented an **Extended Kalman Filter (EKF)** that fuses data from two sources:
- Visual measurements from the camera (position x, y and orientation θ)
- Motor commands and odometry (left and right wheel speeds)

The filter compensates for uncertainties in the process and the measurements and ensures continuous position estimation even when the camera temporarily loses sight of the robot.

### State Model

The kinematic model of the robot can be defined in several different ways.
1. We can consider the robot's position $(x, y, \theta)$ as system states and velocity as inputs. With $A$ being a 3x3 identity matrix, $B$ the control matrix, and $u$ the wheel speeds.
$$\mathbf{x}_{k+1} = \mathbf{A} \mathbf{x}_{k} + \mathbf{B} \mathbf{u}_{k}$$

2. We can also consider velocity as a system state, without external inputs, thus allowing for robot velocity estimation in addition to position. This improves the overall precision of the Kalman filter. $A$ is therefore the 5x5 state transition matrix.
$$\mathbf{x}_{k+1} = \mathbf{A} \mathbf{x}_{k}$$


After discussing with professor Ferrari Trecate Giancarlo giving the multivariable control course (ME-422), we chose to prioritize the second approach by defining the states as follows:

$$\mathbf{x} = \begin{bmatrix} x \\ y \\ \theta \\ v_L \\ v_R \end{bmatrix}$$

where:
- $x, y$ : robot position in mm
- $\theta$ : robot orientation in radians (positive counterclockwise)
- $v_L, v_R$ : left and right wheel speeds in mm/s

### Kinematic Model

The differential robot's kinematic model can be described by the following equations:

$$\mathbf{x}_{k+1} = g(\mathbf{x}_k) = \begin{bmatrix}
x_k + v_{L_k}\Delta t\frac{\cos(\theta)}{2} + v_{R_k}\Delta t\frac{\cos(\theta)}{2} \\
y_k + v_{L_k}\Delta t\frac{\sin(\theta)}{2} + v_{R_k}\Delta t\frac{\sin(\theta)}{2} \\
\theta_k - \Delta t\frac{v_{L_k}}{L} + \Delta t\frac{v_{R_k}}{L}\\
v_{L_k} \\
v_{R_k}
\end{bmatrix} + \mathbf{W}_{k}$$

$$\mathbf{y}_{k} = \mathbf{C} \mathbf{x}_{k} + \mathbf{V}_{k}$$

where:
- $\Delta t$ : time step between two measurements
- $L = 122.7$ mm : robot wheelbase
- $C$ : observation matrix (identity matrix 5x5)
- $W$ : process noise (Gaussian white noise with covariance $Q$)
- $V$ : measurement noise (Gaussian white noise with covariance $R$)

This translates to the following non-linear state transition matrix:
$$\mathbf{A} = \begin{bmatrix}
1 & 0 & 0 & \frac{\Delta t}{2}\cos(\theta) & \frac{\Delta t}{2}\cos(\theta) \\
0 & 1 & 0 & \frac{\Delta t}{2}\sin(\theta) & \frac{\Delta t}{2}\sin(\theta) \\
0 & 0 & 1 & -\frac{\Delta t}{L} & \frac{\Delta t}{L} \\
0 & 0 & 0 & 1 & 0 \\
0 & 0 & 0 & 0 & 1
\end{bmatrix}$$

It can be noted that the model is non-linear due to the trigonometric terms involving the angle $\theta$.
It is therefore necessary to use an Extended Kalman Filter (EKF) to handle this non-linearity. We proceed by linearizing the model at each step around the current state to then work on an approximated linear model. The Jacobian is given by the state transition matrix $J$.

$$\mathbf{J} = \frac{\delta g(\mathbf{x}_k)}{\delta \mathbf{x}_k} = \begin{bmatrix}
1 & 0 & -\sin(\theta)\Delta t\frac{v_L-v_R}{2} & \frac{\Delta t}{2}\cos(\theta) & \frac{\Delta t}{2}\cos(\theta) \\
0 & 1 & \cos(\theta)\Delta t\frac{v_L-v_R}{2} & \frac{\Delta t}{2}\sin(\theta) & \frac{\Delta t}{2}\sin(\theta) \\
0 & 0 & 1 & -\frac{\Delta t}{L} & \frac{\Delta t}{L} \\
0 & 0 & 0 & 1 & 0 \\
0 & 0 & 0 & 0 & 1
\end{bmatrix}$$

The Jacobian is used in the calculation step of the predicted state covariance.

### Covariance Matrices
#### Matrix Q - Process Noise
The $Q$ matrix represents the uncertainty in the motion model:

$$\mathbf{Q} = \begin{bmatrix}
Qx & 0 & 0 & 0 & 0 \\
0 & Qy & 0 & 0 & 0 \\
0 & 0 & Q\theta & 0 & 0 \\
0 & 0 & 0 & Qv_L & 0 \\
0 & 0 & 0 & 0 & Qv_R
\end{bmatrix}$$

The values reflect:
- $Qx$, $Qy$: Position uncertainty: 3.2 mm² in x and y
- $Q\theta$: Orientation uncertainty: 0.214 rad²
- $Qv_L$, $Qv_R$: Speed uncertainty: 0.57 and 0.53 (mm/s)²

The state variances were determined empirically based on behaviors observed during robot tests.
- To determine the variances $Qx$ and $Qy$, we drove the robot in a straight line at constant speed command for 50cm and calculated the standard deviation of the traveled distance measurements.
- To determine the variance $Q\theta$, we rotated the robot on the spot at constant speed command for a 360° angle and calculated the standard deviation of the resulting angle measurements.
- To determine the variances $Qv_L$ and $Qv_R$, we calculated the standard deviation of the robot's speed by dividing the distance covered by the measured time. This was done while the robot was travelling at a constant speed over a fixed distance.

#### Matrix R - Measurement Noise
The **R** matrix adapts according to camera availability:

**With camera**:
$$\mathbf{R}_{camera} = \begin{bmatrix}
Rx & 0 & 0 & 0 & 0 \\
0 & Ry & 0 & 0 & 0 \\
0 & 0 & R\theta & 0 & 0 \\
0 & 0 & 0 & Rvl & 0 \\
0 & 0 & 0 & 0 & Rvr
\end{bmatrix}$$

The values are set as follows:
- $Rx$, $Ry$: Position uncertainty from camera measurements: e.g., 0.349 mm² in x and y
- $R\theta$: Orientation uncertainty from camera measurements: e.g., 0.027 rad²
- $Rvl$, $Rvr$: Wheel speed measurement uncertainty: e.g., 3.46 and 5.05 (mm/s)² (respectively for left and right wheels)

These values are also determined based on sensor characteristics and empirical tuning.
- To determine the variances $Rx$, $Ry$, and $R\theta$, we calculated the standard deviation of the positions measured by the camera with a stationary robot. We placed the robot in a corner of the map to ensure maximum measurement distortion and take the maximal value of standard deviation.
- To determine the variances $Rvl$ and $Rvr$, we calculated the standard deviation of the speeds measured by the robot (on the left and right wheels) during a straight-line run at constant speed command.


**Without camera**:

When the camera is unavailable, position uncertainties are set to infinity, forcing the filter to ignore these measurements and rely only on odometry.
$$\mathbf{R}_{no\ camera} = \begin{bmatrix}
\infty & 0 & 0 & 0 & 0 \\
0 & \infty & 0 & 0 & 0 \\
0 & 0 & \infty & 0 & 0 \\
0 & 0 & 0 & Rvl & 0 \\
0 & 0 & 0 & 0 & Rvr
\end{bmatrix}$$

### Kalman Filter Algorithm

The filter follows the classical Kalman filter steps at each iteration:

#### 1. Prediction
A Priori state and variance estimate
$$\hat{\mathbf{x}}_{k} = \mathbf{A} \mathbf{x}_{k-1}$$
$$\hat{\mathbf{P}}_{k} = \mathbf{J} \mathbf{P}_{k-1} \mathbf{J}^T + \mathbf{Q}$$

#### 2. Innovation
$$\mathbf{i}_k = \mathbf{y}_k - \mathbf{C} \hat{\mathbf{x}}_{k}$$
$$\mathbf{S}_k = \mathbf{C} \hat{\mathbf{P}}_{k} \mathbf{C}^T + \mathbf{R}$$
where $\mathbf{C} = \mathbf{I}_5$ (identity observation matrix since all state components are measured).

#### 3. Kalman Gain
$$\mathbf{K}_k = \hat{\mathbf{P}}_{k} \mathbf{C}^T \mathbf{S}_k^{-1}$$
#### 4. Estimation Update
A Posteriori state and variance estimate
$$\mathbf{x}_{k} = \hat{\mathbf{x}}_{k} + \mathbf{K}_k \mathbf{i}_k$$
$$\mathbf{P}_{k} = (\mathbf{I} - \mathbf{K}_k \mathbf{C}) \hat{\mathbf{P}}_{k}$$

### Implementation

The `Filter` class encapsulates the Kalman filter implementation. It is initialized with the initial position measured by the camera:

```python
filter = Filter(thymio)  # thymio contains [x, y, theta] and are init with a camera measurement
```

At each iteration, the `update()` method is called with:
- `measured_pos` : position measured by the camera (or `None` if unavailable)
- `Ts` : time elapsed since the last update
- `ls, rs` : measured left and right wheel speeds

The filter returns the estimated position `[x, y, theta]` which is used for robot navigation.